In [ ]:
!mkdir /root/.kaggle
!mv kaggle.json /root/.kaggle

In [ ]:
'chmod 600 /root/.kaggle/kaggle.json'

In [ ]:
!kaggle competitions download -c dog-breed-identification

In [ ]:
import os

In [ ]:
!unzip -q dog-breed-identification.zip

In [ ]:
import pandas as pd
df_label=pd.read_csv("labels.csv")

In [ ]:
import cv2
imagenames=[]
height=[]
width=[]
dim=[]
for image in os.listdir("train/"):
      imagenames.append(image)
      img = "train/" + image
      imag = cv2.imread(img)
      height.append(imag.shape[0])
      width.append(imag.shape[1])
      dim.append(imag.shape[2])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
num=np.random.randint(0,10222)
image1=cv2.imread("train/" + imagenames[num])
plt.imshow(image1)

In [ ]:
df_image = pd.DataFrame()
df_image['Filename'] = imagenames


In [ ]:
import statistics as st
print("Average image height",st.mean(height))
print("Average image width",st.mean(width))

In [ ]:
df_label['Filename'] = df_label['id'] + ".jpg"

In [ ]:
df_image=pd.merge(df_image,df_label,how='inner')

In [ ]:
df_image.drop(columns=['id'],inplace=True)

In [ ]:
df_image

In [ ]:
df_image.breed = pd.Categorical(df_image.breed)
df_image['labels'] = df_image.breed.cat.codes

In [ ]:
df_breed = pd.DataFrame(df_image['breed'].value_counts()).reset_index()

In [ ]:
df_breed.column_names=('breed','count')


In [ ]:
df_breed.rename(columns={'index':'breed','breed':'count'}, inplace=True)

In [ ]:
df_breed=df_breed[df_breed['count'] >= 70]

In [ ]:
df_image = pd.merge(df_image,df_breed,how='inner')

In [ ]:
df_image.drop(columns=['count'])

In [ ]:
len(df_image['breed'].unique())

In [ ]:
batchsize=32

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
import tensorflow as tf
idg = tf.keras.preprocessing.image.ImageDataGenerator(horizontal_flip=True,  #aug
                                                      rescale=1/255.0, #normalization
                                                      validation_split=0.1,
                                                      rotation_range=30,
                                                      zoom_range=0.2) #valid

In [ ]:
train_gen=idg.flow_from_dataframe(df_image,directory="train",x_col='Filename',y_col='breed',
                              target_size=(300,300),seed=1,batch_size=batchsize,subset="training")

In [ ]:
val_gen=idg.flow_from_dataframe(df_image,directory="train",x_col='Filename',y_col='breed',
                              target_size=(300,300),seed=1,batch_size=batchsize,subset="validation")

In [ ]:
import tensorflow as tf
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Input(shape=(100, 100, 3)))
# Chunk 1 
model.add(tf.keras.layers.Conv2D(filters=16, kernel_size=(3,3), strides=(1,1), 
                                 activation = tf.keras.activations.relu))
model.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Chunk 2 
model.add(tf.keras.layers.Conv2D(32, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))

model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(256, activation=tf.keras.activations.relu))
model.add(tf.keras.layers.Dense(120, activation=tf.keras.activations.softmax))
model.summary()
model.compile(tf.keras.optimizers.SGD(), 
              loss=tf.keras.losses.categorical_crossentropy, 
              metrics=["acc"])
hist = model.fit(train_gen, batch_size=64, epochs=10, validation_data=val_gen)
model.save("breed_V1.0")

In [ ]:
model1 = tf.keras.models.Sequential()
model1.add(tf.keras.layers.Input(shape = (100, 100,3)))
model1.add(tf.keras.layers.Flatten())
model1.add(tf.keras.layers.Dense(units=1024, activation=tf.keras.activations.softmax))
model1.add(tf.keras.layers.Dense(units=256, activation=tf.keras.activations.softmax))
model1.add(tf.keras.layers.Dense(units=120, activation=tf.keras.activations.softmax))
print(model1.summary())
model1.compile(tf.keras.optimizers.SGD(), 
              loss=tf.keras.losses.categorical_crossentropy, 
              metrics=["acc"])
hist1 = model1.fit(train_gen, batch_size=batchsize, epochs=10, validation_data=val_gen)
model1.save("breed_V1.1")

In [ ]:
import tensorflow as tf
model2 = tf.keras.models.Sequential()
model2.add(tf.keras.layers.Input(shape=(100, 100, 3)))
# Chunk 1 
model2.add(tf.keras.layers.Conv2D(filters=16, kernel_size=(3,3), strides=(1,1), 
                                 activation = tf.keras.activations.relu))
model2.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))  

# Chunk 2 
model2.add(tf.keras.layers.Conv2D(32, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model2.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Chunk 3 
model2.add(tf.keras.layers.Conv2D(64, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model2.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Chunk 4 
model2.add(tf.keras.layers.Conv2D(128, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model2.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))
model2.add(tf.keras.layers.Flatten())
model2.add(tf.keras.layers.Dense(256, activation=tf.keras.activations.relu))
model2.add(tf.keras.layers.Dense(120, activation=tf.keras.activations.softmax))
model2.summary()
model2.compile(tf.keras.optimizers.SGD(), 
              loss=tf.keras.losses.categorical_crossentropy, 
              metrics=["acc"])
hist2 = model2.fit(train_gen, batch_size=32, epochs=25, validation_data=val_gen)

In [ ]:
import tensorflow as tf
model3 = tf.keras.models.Sequential()
model3.add(tf.keras.layers.Input(shape=(300,300, 3)))
# Chunk 1 
model3.add(tf.keras.layers.Conv2D(filters=8, kernel_size=(3,3), strides=(1,1), 
                                 activation = tf.keras.activations.relu))
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))  

# Chunk 2 
model3.add(tf.keras.layers.Conv2D(16, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Chunk 3 
model3.add(tf.keras.layers.Conv2D(32, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Chunk 4 
model3.add(tf.keras.layers.Conv2D(64, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))


# Chunk 5 
model3.add(tf.keras.layers.Conv2D(128, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))






model3.add(tf.keras.layers.Flatten())
model3.add(tf.keras.layers.Dense(256, activation=tf.keras.activations.relu))
model3.add(tf.keras.layers.Dense(110, activation=tf.keras.activations.softmax))
model3.summary()
model3.compile(tf.keras.optimizers.SGD(), 
              loss=tf.keras.losses.categorical_crossentropy, 
              metrics=["acc"])
hist3 = model3.fit(train_gen, batch_size=32, epochs=25, validation_data=val_gen)

In [ ]:
from tensorflow.keras import applications
vgg = applications.VGG16(include_top=True)
vgg.summary()

In [ ]:
vgg2 = applications.VGG16(include_top=False, input_shape=(300, 300, 3))
vgg2.summary()

In [ ]:
for layer in vgg2.layers:
  layer.trainable = False

In [ ]:
flat = tf.keras.layers.Flatten() (vgg2.output) 
d1 = tf.keras.layers.Dense(256, activation='relu') (flat)
pred = tf.keras.layers.Dense(110, activation='softmax') (d1)

In [ ]:
model6 = tf.keras.Model(inputs=[vgg2.input], outputs=[pred])

In [ ]:
model6.compile(tf.keras.optimizers.SGD(), loss=tf.keras.losses.categorical_crossentropy, metrics=["acc"]) 

In [ ]:
model6.fit(train_gen, validation_data=val_gen, epochs=25, batch_size=batchsize)    

In [ ]:
from tensorflow.keras import applications
inc = applications.InceptionV3(include_top=True)
inc.summary()

In [ ]:
for layer in inc.layers:
  layer.trainable = False

In [ ]:
flat = tf.keras.layers.Flatten() (inc.output) 
d1 = tf.keras.layers.Dense(256, activation='relu') (flat)
pred = tf.keras.layers.Dense(110, activation='softmax') (d1)

In [ ]:
final_model = tf.keras.Model(inputs=[inc.input], outputs=[pred])

In [ ]:
final_model.compile(tf.keras.optimizers.SGD(), loss=tf.keras.losses.categorical_crossentropy, metrics=["acc"]) 

In [ ]:
final_model.fit(train_gen, validation_data=val_gen, epochs=25, batch_size=32) 

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D
from tensorflow.keras.optimizers import Adam

train_gen=idg.flow_from_dataframe(df_image,directory="train",x_col='Filename',y_col='breed',
                              target_size=(224,224),seed=1,batch_size=batchsize,subset="training")
val_gen=idg.flow_from_dataframe(df_image,directory="train",x_col='Filename',y_col='breed',
                              target_size=(224,224),seed=1,batch_size=batchsize,subset="validation")

model = Sequential()

model.add(Conv2D(filters = 64, kernel_size = (5,5), activation ='relu', input_shape = (224,224,3)))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters = 32, kernel_size = (3,3), activation ='relu', kernel_regularizer = 'l2'))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters = 16, kernel_size = (7,7), activation ='relu', kernel_regularizer = 'l2'))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters = 8, kernel_size = (5,5), activation ='relu', kernel_regularizer = 'l2'))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Flatten())
model.add(Dense(128, activation = "relu", kernel_regularizer = 'l2'))
model.add(Dense(64, activation = "relu", kernel_regularizer = 'l2'))
model.add(Dense(110, activation = "softmax"))
model.summary()


In [ ]:
model.compile(loss = 'categorical_crossentropy', optimizer = Adam(0.0001),metrics=['accuracy'])
model.fit(train_gen, validation_data=val_gen, epochs=25, batch_size=32) 

In [ ]:
import tensorflow as tf
model3 = tf.keras.models.Sequential()
model3.add(tf.keras.layers.Input(shape=(300,300, 3)))
# Chunk 1 
model3.add(tf.keras.layers.Conv2D(filters=128, kernel_size=(3,3), strides=(1,1), 
                                 activation = tf.keras.activations.relu))
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))  

# Chunk 2 
model3.add(tf.keras.layers.Conv2D(64, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Chunk 3 
model3.add(tf.keras.layers.Conv2D(32, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Chunk 4 
model3.add(tf.keras.layers.Conv2D(16, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))


# Chunk 5 
model3.add(tf.keras.layers.Conv2D(8, kernel_size=(3,3), strides=(1,1), 
                                 activation=tf.keras.activations.relu))
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=(2,2)))






model3.add(tf.keras.layers.Flatten())
model3.add(tf.keras.layers.Dense(256, activation=tf.keras.activations.relu))
model3.add(tf.keras.layers.Dense(110, activation=tf.keras.activations.softmax))
model3.summary()
model3.compile(tf.keras.optimizers.SGD(), 
              loss=tf.keras.losses.categorical_crossentropy, 
              metrics=["acc"])
hist3 = model3.fit(train_gen, batch_size=32, epochs=25, validation_data=val_gen)